# Phase 2: verify pipeline.search() live, tune tau_high/tau_low

Three Phase 2 checklist items in one run, since they all need the same live
`pipeline.search()` calls against real Qdrant + real models:

1. **Package retrieval as a module, models loaded once** -- verify a second
   query returns under 1s and memory is flat across 100 requests.
2. **Make the English leg conditional on the Sindhi top score** -- compare
   median latency and Recall@1 with the leg forced off vs conditional.
3. **Tune tau_high and tau_low** -- from the 275-query corrected gold set
   (correct/incorrect top-1 scores) and the 100-query negative set (scores
   for queries with no correct KB answer).

**Runtime -> Change runtime type -> T4 GPU.**

In [ ]:
!pip install -q qdrant-client FlagEmbedding transformers sentencepiece psutil matplotlib

In [ ]:
import os

work_dir = "/kaggle/working" if os.path.isdir("/kaggle") else "/content"
%cd $work_dir
!rm -rf naari-ai
!git clone --branch sana/test-protection --depth 1 https://github.com/sana200420/naari-ai.git
%cd naari-ai

import sys
sys.path.insert(0, ".")

print("cloned OK")

In [ ]:
import os
from getpass import getpass

# pipeline.py reads these from os.environ directly (12-factor style) --
# set them as real env vars, not just local Python variables.
os.environ["QDRANT_URL"] = getpass("Qdrant cluster URL: ")
os.environ["QDRANT_API_KEY"] = getpass("Qdrant API key: ")

print("env vars set")

In [ ]:
import csv

with open("eval/gold_eval_280_linked.csv", encoding="utf-8-sig", newline="") as f:
    gold_rows = list(csv.DictReader(f))

with open("eval/negative_set_100.csv", encoding="utf-8-sig", newline="") as f:
    negative_rows = list(csv.DictReader(f))

with open("knowledge_base/Womens_Health_KB - 2000_final.csv", encoding="utf-8", newline="") as f:
    kb_by_id = {r["id"]: r for r in csv.DictReader(f)}

# acceptable_answer_ids holds a small set of KB ids that should all grade as
# correct for a query (seeded from manually-verified near-duplicate KB rows --
# see eval/gold_eval_280_linked.csv's acceptable_answer_ids column). Falls back
# to just correct_answer_id for rows that don't have the column populated yet.
for row in gold_rows:
    raw = row.get("acceptable_answer_ids", "").strip()
    row["_acceptable_ids"] = set(raw.split(";")) if raw else {row["correct_answer_id"]}

print(f"{len(gold_rows)} gold rows (expect 248), {len(negative_rows)} negative rows (expect 100), "
      f"{len(kb_by_id)} KB rows loaded")
assert len(gold_rows) == 248
assert len(negative_rows) == 100

## Item 1: warm-loading, latency, memory

First call pays every model's load cost (bge-m3, bge-reranker-v2-m3, and
NLLB *only if* the first query happens to be uncertain enough to trigger the
English leg). Second call, and every call after, should be warm.

In [ ]:
import time

from retrieval.pipeline import search, warmup

# Force every model to load now (embed, rerank, AND translate) instead of
# letting translate's NLLB load lazily whenever the first uncertain query
# happens to show up -- that was the bug the first run of this notebook
# hit: query 1 didn't need the English leg, so query 2 silently paid
# NLLB's ~60s cold-load cost and failed the "under 1s" check below.
t0 = time.perf_counter()
warmup()
warmup_ms = (time.perf_counter() - t0) * 1000
print(f"warmup(): {warmup_ms:.0f}ms (loads bge-m3, bge-reranker-v2-m3, NLLB, and connects to Qdrant)")

t0 = time.perf_counter()
first = search(gold_rows[0]["query"])
first_ms = (time.perf_counter() - t0) * 1000
print(f"first real query (post-warmup): {first_ms:.0f}ms -- reported latency_ms: {first['latency_ms']}")

t0 = time.perf_counter()
second = search(gold_rows[1]["query"])
warm_ms = (time.perf_counter() - t0) * 1000
print(f"second real query: {warm_ms:.0f}ms -- reported latency_ms: {second['latency_ms']}")

if warm_ms >= 1000:
    print(f"\nWARNING: second call took {warm_ms:.0f}ms, expected under 1000ms -- "
          "not halting, see the diagnostic cell below for a step-by-step breakdown.")

## Diagnostic: which step is actually slow?

warmup() genuinely loaded everything this run (177s -- a real fresh
download+load, unlike the first attempt's suspiciously-fast 3s that turned
out to be stale kernel state). But the two real queries right after it are
*still* slow, which means this isn't the same lazy-loading bug anymore --
something else is the bottleneck. Times every sub-step of `search()`
individually instead of guessing again.

In [ ]:
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"device: {torch.cuda.get_device_name(0)}")

import retrieval.translate as translate_module
import retrieval.embed as embed_module
import retrieval.rerank as rerank_module
from retrieval.pipeline import _get_retriever

print(f"translate module's detected device: {translate_module._device}")

retriever = _get_retriever()
query = gold_rows[2]["query"]

t0 = time.perf_counter()
embed_module.embed_text(query)
print(f"embed_text: {(time.perf_counter() - t0) * 1000:.0f}ms")

t0 = time.perf_counter()
dense_rows = retriever.dense_search(query, top_k=20, lang="sd")
print(f"dense_search (sd): {(time.perf_counter() - t0) * 1000:.0f}ms")

t0 = time.perf_counter()
sparse_rows = retriever.sparse_search(query, top_k=20, lang="sd")
print(f"sparse_search (sd): {(time.perf_counter() - t0) * 1000:.0f}ms")

candidates = dense_rows[:5] if dense_rows else []
t0 = time.perf_counter()
rerank_module.rerank(query, candidates, top_k=5)
print(f"rerank ({len(candidates)} candidates): {(time.perf_counter() - t0) * 1000:.0f}ms")

t0 = time.perf_counter()
en_query = translate_module.translate_sd_to_en(query)
print(f"translate_sd_to_en: {(time.perf_counter() - t0) * 1000:.0f}ms -> '{en_query}'")

t0 = time.perf_counter()
retriever.dense_search(en_query, top_k=20, lang="en")
print(f"dense_search (en): {(time.perf_counter() - t0) * 1000:.0f}ms")

In [ ]:
import psutil

process = psutil.Process(os.getpid())
mem_samples = []

for i, row in enumerate(gold_rows[:100], start=1):
    search(row["query"])
    if i % 10 == 0:
        rss_mb = process.memory_info().rss / (1024 ** 2)
        mem_samples.append((i, rss_mb))
        print(f"{i}/100 -- RSS: {rss_mb:.0f}MB")

first_half = [m for i, m in mem_samples if i <= 50]
second_half = [m for i, m in mem_samples if i > 50]
growth_mb = (sum(second_half) / len(second_half)) - (sum(first_half) / len(first_half))
print(f"\nmean RSS first 50 calls: {sum(first_half)/len(first_half):.0f}MB, "
      f"last 50: {sum(second_half)/len(second_half):.0f}MB, growth: {growth_mb:+.0f}MB")
print("flat (no leak) if growth is small relative to baseline RSS, not a per-call increment.")

## Item 3 setup: collect top-1 scores for tau tuning

Runs every gold query and every negative-set query through the real
pipeline once (default `tau_high` from the `TAU_HIGH` env var / 0.75
fallback), recording the top-1 score and, for gold queries, whether it was
correct.

In [ ]:
from retrieval.pipeline import _get_retriever

retriever = _get_retriever()

gold_scored = []
fused_top20_cache = {}  # query_id -> [answer_id, ...] in fused-search rank order

for i, row in enumerate(gold_rows, start=1):
    query = row["query"]
    gt_ids = row["_acceptable_ids"]

    # Pre-rerank fused top-20 -- this is the actual retrieval shortlist, used for
    # Recall@20 (is the answer in the pool at all?) and hub-row frequency counting.
    fused_rows = retriever.fused_search(query, top_k=20, leg_k=25, lang="sd")
    fused_ids = [str(r["answer_id"]) for r in fused_rows]
    fused_top20_cache[row["query_id"]] = fused_ids
    correct_rank = next((rank for rank, aid in enumerate(fused_ids, start=1) if aid in gt_ids), None)

    # Full pipeline (rerank + conditional English leg), top-2 so we get a margin
    # (top1 score - top2 score) for free alongside the final top-1.
    result = search(query, top_k=2)
    results = result["results"]
    top1 = results[0] if results else None
    top2 = results[1] if len(results) > 1 else None
    top1_id = str(top1["answer_id"]) if top1 else None

    gold_scored.append({
        "query_id": row["query_id"],
        "correct_answer_id": row["correct_answer_id"],
        "stated_category": row.get("stated_category", ""),
        "top1_id": top1_id,
        "predicted_category": kb_by_id.get(top1_id, {}).get("category") if top1_id else None,
        "score": top1["score"] if top1 else 0.0,
        "margin": (top1["score"] - top2["score"]) if (top1 and top2) else (top1["score"] if top1 else 0.0),
        "correct": bool(top1) and top1_id in gt_ids,
        "correct_in_top20": correct_rank is not None,
        "correct_rank_in_top20": correct_rank,
        "fusion_agrees_with_final": bool(fused_ids) and bool(top1_id) and fused_ids[0] == top1_id,
    })
    if i % 40 == 0:
        print(f"gold {i}/{len(gold_rows)}")

print(f"done, {len(gold_scored)} gold queries scored")

In [ ]:
negative_scored = []
for i, row in enumerate(negative_rows, start=1):
    result = search(row["question"], top_k=1)
    top1 = result["results"][0] if result["results"] else None
    negative_scored.append({
        "id": row["id"],
        "score": top1["score"] if top1 else 0.0,
    })
    if i % 25 == 0:
        print(f"negative {i}/{len(negative_rows)}")

print(f"done, {len(negative_scored)} negative queries scored")

## Item 3: pick tau_high and tau_low

tau_high = smallest score threshold where precision("top-1 is correct") on
the gold set reaches >=0.95. tau_low = the score below which >=90% of the
negative set's own top-1 scores fall, per docs/PLAYBOOKS.md Lever 5.

In [ ]:
sorted_by_score = sorted(gold_scored, key=lambda r: -r["score"])

tau_high = None
for cutoff_row in sorted_by_score:
    threshold = cutoff_row["score"]
    at_or_above = [r for r in gold_scored if r["score"] >= threshold]
    precision = sum(r["correct"] for r in at_or_above) / len(at_or_above)
    if precision >= 0.95:
        tau_high = threshold
        coverage = len(at_or_above) / len(gold_scored)

print(f"tau_high = {tau_high:.4f}" if tau_high else "no threshold reaches 0.95 precision anywhere")
if tau_high:
    print(f"at this threshold: precision={precision:.3f}, coverage={coverage:.3f} "
          f"({int(coverage*len(gold_scored))}/{len(gold_scored)} gold queries clear it)")

In [ ]:
negative_scores_sorted = sorted(r["score"] for r in negative_scored)
p90_index = int(0.90 * len(negative_scores_sorted))
tau_low = negative_scores_sorted[p90_index]

below = sum(1 for s in negative_scores_sorted if s < tau_low)
print(f"tau_low = {tau_low:.4f} -- {below}/{len(negative_scores_sorted)} "
      f"({below/len(negative_scores_sorted):.1%}) of the negative set falls below it")

if tau_high and tau_low >= tau_high:
    print("\nWARNING: tau_low >= tau_high -- the two bands overlap or invert. "
          "This means correct and negative-set scores aren't cleanly separated; "
          "don't ship these values without a closer look.")

In [ ]:
import matplotlib.pyplot as plt

correct_scores = [r["score"] for r in gold_scored if r["correct"]]
incorrect_scores = [r["score"] for r in gold_scored if not r["correct"]]
neg_scores = [r["score"] for r in negative_scored]

fig, ax = plt.subplots(figsize=(9, 5))
bins = 30
ax.hist(correct_scores, bins=bins, alpha=0.6, label=f"gold: correct top-1 (n={len(correct_scores)})", color="#2a9d8f")
ax.hist(incorrect_scores, bins=bins, alpha=0.6, label=f"gold: incorrect top-1 (n={len(incorrect_scores)})", color="#e76f51")
ax.hist(neg_scores, bins=bins, alpha=0.6, label=f"negative set (n={len(neg_scores)})", color="#264653")
if tau_high:
    ax.axvline(tau_high, color="#2a9d8f", linestyle="--", label=f"tau_high = {tau_high:.3f}")
ax.axvline(tau_low, color="#264653", linestyle="--", label=f"tau_low = {tau_low:.3f}")
ax.set_xlabel("pipeline.search() top-1 score")
ax.set_ylabel("count")
ax.set_title("Score distributions: correct vs incorrect (gold) vs negative set")
ax.legend()
fig.tight_layout()
fig.savefig("eval/tau_score_distributions.png", dpi=150)
plt.show()
print("saved eval/tau_score_distributions.png")

## Diagnostics (added 2026-09-02, before any further tau tuning)

Four checks requested before trusting tau_high work any further:
1. **Recall@20 vs Recall@1** — is the correct answer usually *in* the shortlist
   (ranking problem — fix reranking/fusion) or usually *missing* from it
   entirely (retrieval problem — fix embedding/indexing, reranking work would
   be wasted)?
2. **Hub-row frequency** — which KB rows dominate shortlists across unrelated
   queries (the id=610/821/etc. pattern seen in three review passes).
3. **Structured audit** of every incorrect-but-score>=0.9 query, with fields
   that route to a decision rather than just a description.
4. **Alternative confidence signals** (margin, fusion/rerank agreement,
   category consistency) vs. the raw score alone, fit on a train split and
   reported on a held-out test split.

In [ ]:
import statistics

def run_pass(tau):
    latencies, correct = [], 0
    for row in gold_rows:
        t0 = time.perf_counter()
        result = search(row["query"], top_k=1, tau_high=tau)
        latencies.append((time.perf_counter() - t0) * 1000)
        top1 = result["results"][0] if result["results"] else None
        if top1 and str(top1["answer_id"]) in row["_acceptable_ids"]:
            correct += 1
    return latencies, correct / len(gold_rows)

sd_only_latencies, sd_only_recall = run_pass(tau=0.0)
print(f"English leg forced OFF (tau_high=0.0) -- median latency: {statistics.median(sd_only_latencies):.0f}ms, "
      f"Recall@1: {sd_only_recall:.3f}")

conditional_latencies, conditional_recall = run_pass(tau=tau_high or 0.75)
print(f"conditional (tau_high={tau_high or 0.75:.3f}) -- median latency: "
      f"{statistics.median(conditional_latencies):.0f}ms, Recall@1: {conditional_recall:.3f}")

# The checklist item actually asks: does making the leg CONDITIONAL save time
# over running it UNCONDITIONALLY? That's conditional vs always-on, not
# conditional vs forced-off (forced-off is a different, already-answered
# question -- the Lever 4 rescue-rate result).
always_on_latencies, always_on_recall = run_pass(tau=1.1)
print(f"English leg ALWAYS ON (tau_high=1.1) -- median latency: "
      f"{statistics.median(always_on_latencies):.0f}ms, Recall@1: {always_on_recall:.3f}")

print(f"\nmedian latency saved by being conditional instead of always-on: "
      f"{statistics.median(always_on_latencies) - statistics.median(conditional_latencies):.0f}ms")

In [ ]:
import datetime

lines = []
lines.append("\n\n# Phase 2 -- pipeline verification, tau tuning, and the post-review diagnostics\n")
lines.append(f"Generated: {datetime.datetime.utcnow().isoformat()}Z, via retrieval/scripts/verify_pipeline_and_tune_thresholds.ipynb\n\n")
lines.append(f"Gold set: {len(gold_rows)} rows (fully individually reviewed, see docs/status.md).\n\n")

lines.append("## Item 1 -- warm loading, latency, memory\n")
lines.append(f"Explicit `warmup()` (loads bge-m3, bge-reranker-v2-m3, NLLB, connects to Qdrant): {warmup_ms:.0f}ms. "
              f"First real query post-warmup: {first_ms:.0f}ms. Second real query: {warm_ms:.0f}ms "
              f"(target: under 1000ms). Memory across 100 calls: first-50 mean {sum(first_half)/len(first_half):.0f}MB, "
              f"last-50 mean {sum(second_half)/len(second_half):.0f}MB, growth {growth_mb:+.0f}MB.\n\n")

lines.append("## Item 2 -- conditional English leg (forced off / conditional / always-on)\n")
lines.append("| Mode | Median latency | Recall@1 |\n|---|---:|---:|\n")
lines.append(f"| Forced off (tau_high=0.0) | {statistics.median(sd_only_latencies):.0f}ms | {sd_only_recall:.3f} |\n")
lines.append(f"| Conditional (tau_high={tau_high or 0.75:.3f}) | {statistics.median(conditional_latencies):.0f}ms | {conditional_recall:.3f} |\n")
lines.append(f"| Always on (tau_high=1.1) | {statistics.median(always_on_latencies):.0f}ms | {always_on_recall:.3f} |\n\n")
lines.append(f"Median latency saved by conditional vs. always-on (the actual question this item asks): "
              f"{statistics.median(always_on_latencies) - statistics.median(conditional_latencies):.0f}ms.\n\n")

lines.append("## Item 3 -- tau_high / tau_low\n")
lines.append(f"**tau_high = {tau_high:.4f}**" if tau_high else "**tau_high: no threshold reached 0.95 precision**")
if tau_high:
    lines.append(f" -- precision {precision:.3f}, coverage {coverage:.3f} "
                  f"({int(coverage*len(gold_scored))}/{len(gold_scored)} gold queries)\n\n")
else:
    lines.append("\n\n")
lines.append(f"**tau_low = {tau_low:.4f}** -- {below}/{len(negative_scores_sorted)} "
              f"({below/len(negative_scores_sorted):.1%}) of the negative set falls below it (target >=90%).\n\n")
lines.append("Score distribution figure: `eval/tau_score_distributions.png`.\n\n")

lines.append("## Diagnostic 1 -- Recall@20 vs Recall@1\n")
lines.append(f"Recall@1 (final pipeline): {recall_at_1:.3f}. Recall@20 (fused shortlist, pre-rerank): {recall_at_20:.3f}. "
              f"{len(missed_entirely)}/{len(gold_scored)} queries never surface the correct answer anywhere in the "
              "top-20 shortlist at all -- a retrieval-pool gap, not something reranking or tau tuning can fix.\n\n")

lines.append("## Diagnostic 2 -- hub-row frequency\n")
if hub_rows:
    lines.append(f"{len(hub_rows)} KB row(s) appear in over 20% of all {n_queries} shortlists:\n\n")
    for aid, count in hub_rows:
        kb_row = kb_by_id.get(aid, {})
        lines.append(f"- id={aid} ({count/n_queries:.0%} of shortlists) [{kb_row.get('category', '?')}]: {kb_row.get('question', '?')}\n")
    lines.append("\n")
else:
    lines.append("No KB row exceeded the 20% shortlist-frequency threshold this run.\n\n")

lines.append("## Diagnostic 3 -- structured audit of high-scoring wrong answers\n")
lines.append(f"{len(audit_rows)} incorrect-but-score>=0.9 queries logged to `eval/high_score_wrong_answer_audit.csv` "
              "(human columns left blank for a follow-up pass). Before that pass: "
              f"{sum(1 for a in audit_rows if a['category_mismatch'])}/{len(audit_rows) or 1} have a category mismatch, "
              f"{sum(1 for a in audit_rows if a['shares_distinctive_phrase'])}/{len(audit_rows) or 1} share a distinctive "
              f"3-word phrase with the wrong answer, {sum(1 for a in audit_rows if a['correct_in_top20'])}/{len(audit_rows) or 1} "
              "have the correct answer sitting somewhere else in the top-20 (a ranking problem, not a pool problem).\n\n")

lines.append("## Diagnostic 4 -- alternative confidence signals (train/test split)\n")
lines.append("See notebook output for the full table (raw score vs. margin vs. fusion/rerank agreement vs. "
              "category consistency, fit on a 70% train split and reported on a held-out 30% test split).\n\n")

lines.append("## Closure -- English leg: broken translations or genuinely unhelpful?\n")
if sindhi_misses:
    lines.append(f"{len(sindhi_misses)} queries never surface the correct answer in the Sindhi-only top-20 "
                  f"(the real rescue candidates). Of {len(sample)} sampled, the English leg rescued {rescued} "
                  "-- see notebook output for the actual NLLB translations.\n\n")
else:
    lines.append("No Sindhi-only misses this run -- nothing to check.\n\n")

with open("eval/results.md", "a", encoding="utf-8") as f:
    f.write("".join(lines))

print("appended to eval/results.md")
print("".join(lines))

In [ ]:
import os

outputs = ["eval/results.md", "eval/tau_score_distributions.png", "eval/high_score_wrong_answer_audit.csv"]
outputs = [p for p in outputs if os.path.exists(p)]

try:
    from google.colab import files
    for p in outputs:
        files.download(p)
except ImportError:
    # Kaggle (or anywhere else without google.colab): files are already sitting
    # under the working directory -- grab them from the notebook's Output/file
    # browser after the run finishes.
    print("Not on Colab -- files were not auto-downloaded. Grab these from the "
          "notebook's file browser (Kaggle: the Output tab) instead:")
    for p in outputs:
        print(" ", os.path.abspath(p))

In [ ]:
# Diagnostic 4: is the raw score even the right confidence signal? Compare it
# against margin (top1-top2), fusion/rerank agreement, and category consistency.
# Fit each candidate threshold on a 70% train split, report on the 30% held-out
# test split -- tuning and evaluating on the same 248 rows is exactly what
# produced an over-fit-looking tau_high before.
import random

labeled = list(gold_scored)
random.seed(42)
random.shuffle(labeled)
split = int(0.7 * len(labeled))
train, test = labeled[:split], labeled[split:]


def precision_coverage(rows, predicate):
    selected = [r for r in rows if predicate(r)]
    if not selected:
        return None, 0.0
    return sum(r["correct"] for r in selected) / len(selected), len(selected) / len(rows)


def best_threshold(train_rows, key):
    best = (None, 0.0, 0.0)  # threshold, precision, coverage
    for r in sorted(train_rows, key=lambda x: -key(x)):
        t = key(r)
        p, c = precision_coverage(train_rows, lambda x: key(x) >= t)
        if p is not None and p >= 0.95 and c > best[2]:
            best = (t, p, c)
    return best


print("Fit on train (70%), report on held-out test (30%):\n")
for name, key in [("raw score", lambda r: r["score"]), ("margin (top1-top2)", lambda r: r["margin"])]:
    t, train_p, train_c = best_threshold(train, key)
    if t is None:
        print(f"{name}: no threshold hit 0.95 precision on train")
        continue
    test_p, test_c = precision_coverage(test, lambda r: key(r) >= t)
    test_p_str = f"{test_p:.3f}" if test_p is not None else "n/a"
    print(f"{name}: threshold={t:.4f} -- train precision={train_p:.3f}/coverage={train_c:.3f}, "
          f"test precision={test_p_str}/coverage={test_c:.3f}")

print()
for name, predicate in [
    ("fusion agrees with final top-1", lambda r: r["fusion_agrees_with_final"]),
    ("predicted category == stated category", lambda r: r["predicted_category"] == r["stated_category"]),
    ("agreement AND category match", lambda r: r["fusion_agrees_with_final"] and r["predicted_category"] == r["stated_category"]),
]:
    p, c = precision_coverage(test, predicate)
    p_str = f"{p:.3f}" if p is not None else "n/a"
    print(f"{name}: test precision={p_str}, coverage={c:.3f}")

## Closure: is the English leg actually broken, or just not useful?

For every query that never surfaces its correct answer anywhere in the
Sindhi-only top-20 (the real candidates for cross-lingual rescue), print the
NLLB translation and check whether the English leg's own top-20 finds it.
Bad translations -> the leg is broken and fixable. Good translations that
still miss -> there's genuinely no better match in English; delete the leg
and reclaim the latency/RAM.

In [ ]:
from retrieval.translate import translate_sd_to_en

sindhi_misses = [r for r in gold_scored if not r["correct_in_top20"]]
print(f"{len(sindhi_misses)} queries never surface the correct answer in the Sindhi-only "
      "top-20 -- these are the real candidates for English-leg rescue. Sampling up to 20:\n")

sample = sindhi_misses[:20]
rescued = 0
for r in sample:
    gold_row = gold_by_id[r["query_id"]]
    query = gold_row["query"]
    en_query = translate_sd_to_en(query)
    en_rows = retriever.dense_search(en_query, top_k=20, lang="en")
    en_ids = {str(er["answer_id"]) for er in en_rows}
    hit = bool(en_ids & gold_row["_acceptable_ids"])
    rescued += hit
    print(f"[{'RESCUED' if hit else 'missed'}] {r['query_id']}")
    print(f"  SD: {query}")
    print(f"  EN translation: {en_query}\n")

if sample:
    print(f"\n{rescued}/{len(sample)} sampled Sindhi-only misses rescued by the English leg.")
    print("Read the translations above: garbled/wrong translations -> the leg is broken, "
          "worth fixing. Accurate translations that still miss -> the leg genuinely isn't "
          "earning its latency/RAM cost on this KB -- worth deleting.")
else:
    print("no Sindhi-only misses this run -- nothing to check.")

## Item 2: conditional English leg -- does it actually save latency without costing recall?

Re-runs the gold set twice: once with the English leg forced off
(`tau_high=0.0`, so the Sindhi-only score always already "clears" the bar),
once with the tuned `tau_high` from above (conditional -- only translates
when genuinely uncertain).

In [ ]:
import statistics

def run_pass(tau):
    latencies, correct = [], 0
    for row in gold_rows:
        t0 = time.perf_counter()
        result = search(row["query"], top_k=1, tau_high=tau)
        latencies.append((time.perf_counter() - t0) * 1000)
        top1 = result["results"][0] if result["results"] else None
        if top1 and str(top1["answer_id"]) == str(row["correct_answer_id"]):
            correct += 1
    return latencies, correct / len(gold_rows)

sd_only_latencies, sd_only_recall = run_pass(tau=0.0)
print(f"English leg forced OFF -- median latency: {statistics.median(sd_only_latencies):.0f}ms, "
      f"Recall@1: {sd_only_recall:.3f}")

conditional_latencies, conditional_recall = run_pass(tau=tau_high or 0.75)
print(f"conditional (tau_high={tau_high or 0.75:.3f}) -- median latency: "
      f"{statistics.median(conditional_latencies):.0f}ms, Recall@1: {conditional_recall:.3f}")

In [ ]:
import datetime

lines = []
lines.append("\n\n# Phase 2 -- pipeline verification and tau tuning\n")
lines.append(f"Generated: {datetime.datetime.utcnow().isoformat()}Z, via retrieval/scripts/verify_pipeline_and_tune_thresholds.ipynb\n\n")

lines.append("## Item 1 -- warm loading, latency, memory\n")
lines.append(f"Explicit `warmup()` (loads bge-m3, bge-reranker-v2-m3, NLLB, connects to Qdrant): {warmup_ms:.0f}ms. "
              f"First real query post-warmup: {first_ms:.0f}ms. Second real query: {warm_ms:.0f}ms "
              f"(target: under 1000ms). Memory across 100 calls: first-50 mean {sum(first_half)/len(first_half):.0f}MB, "
              f"last-50 mean {sum(second_half)/len(second_half):.0f}MB, growth {growth_mb:+.0f}MB.\n\n")

lines.append("## Item 2 -- conditional English leg\n")
lines.append("| Mode | Median latency | Recall@1 |\n|---|---:|---:|\n")
lines.append(f"| English leg forced off | {statistics.median(sd_only_latencies):.0f}ms | {sd_only_recall:.3f} |\n")
lines.append(f"| Conditional (tau_high={tau_high or 0.75:.3f}) | {statistics.median(conditional_latencies):.0f}ms | {conditional_recall:.3f} |\n\n")

lines.append("## Item 3 -- tau_high / tau_low\n")
lines.append(f"**tau_high = {tau_high:.4f}**" if tau_high else "**tau_high: no threshold reached 0.95 precision**")
if tau_high:
    lines.append(f" -- precision {precision:.3f}, coverage {coverage:.3f} "
                  f"({int(coverage*len(gold_scored))}/{len(gold_scored)} gold queries)\n\n")
else:
    lines.append("\n\n")
lines.append(f"**tau_low = {tau_low:.4f}** -- {below}/{len(negative_scores_sorted)} "
              f"({below/len(negative_scores_sorted):.1%}) of the negative set falls below it (target >=90%).\n\n")
lines.append("Score distribution figure: `eval/tau_score_distributions.png`.\n")

with open("eval/results.md", "a", encoding="utf-8") as f:
    f.write("".join(lines))

print("appended to eval/results.md")
print("".join(lines))

In [ ]:
from google.colab import files
files.download("eval/results.md")
files.download("eval/tau_score_distributions.png")